In [1]:
import math
import warnings
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb

import optuna
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import LinearSVR
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import root_mean_squared_error

import mlflow
from mlflow import MlflowClient
from mlflow.models import infer_signature

# feature extraction tools
from mlops_zoomcamp.features import add_features

In [2]:
EXPERIMENT_NAME = "nyc-taxi-experiment-v1"
REGISTERED_MODEL_NAME = "nyc-taxi-duration"

# skops (MLflow's default format) refuses non-sklearn types unless trusted;
# code_paths ships the module with the model so it loads anywhere
MODEL_LOG_KWARGS = dict(
    skops_trusted_types=[
        'mlops_zoomcamp.features.add_features',
        'xgboost.sklearn.XGBRegressor',
        'xgboost.core.Booster',
        'sklearn.tree._tree.Tree',  # node storage of DecisionTree / RandomForest
        'sklearn.neural_network._stochastic_optimizers.AdamOptimizer',  # MLPRegressor solver state
    ],
    code_paths=['../src/mlops_zoomcamp'],
)

# Location IDs are int32 and never null in TLC data, so MLflow's "integer column" schema hint doesn't apply
warnings.filterwarnings('ignore', message='Hint: Inferred schema contains integer column')

mlflow.set_tracking_uri("sqlite:///../mlflow/mlflow.db")

# create_experiment fails if the experiment exists, so only create it once
if mlflow.get_experiment_by_name(EXPERIMENT_NAME) is None:
    mlflow.create_experiment(EXPERIMENT_NAME, artifact_location="file:../mlflow")
mlflow.set_experiment(EXPERIMENT_NAME)

# Autolog records params and training metrics of every sklearn fit into the active run.
# Models are logged explicitly instead: autolog can't pass code_paths / skops_trusted_types.
# Datasets are logged explicitly too, with their file sources.
mlflow.sklearn.autolog(log_models=False, log_datasets=False, silent=True)
optuna.logging.set_verbosity(optuna.logging.WARNING)


2026/10/16 20:04:36 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/16 20:04:36 INFO mlflow.store.db.utils: Updating database tables


# Data

In [3]:
def read_dataframe(filename):
    if filename.endswith('.csv'):
        df = pd.read_csv(filename, parse_dates=['lpep_pickup_datetime', 'lpep_dropoff_datetime'])
    else:
        df = pd.read_parquet(filename)

    df['duration'] = (df.lpep_dropoff_datetime - df.lpep_pickup_datetime).dt.total_seconds() / 60
    return df[(df.duration >= 1) & (df.duration <= 60)].copy()

train_path = './data/green_tripdata_2026-01.parquet'
val_path = './data/green_tripdata_2026-02.parquet'

df_train = read_dataframe(train_path)
df_val = read_dataframe(val_path)

# Raw columns only: feature engineering lives inside the pipeline,
# so the logged model accepts the same input at serving time
raw_cols = ['PULocationID', 'DOLocationID', 'trip_distance']
target = 'duration'

X_train, y_train = df_train[raw_cols], df_train[target]
X_val, y_val = df_val[raw_cols], df_val[target]

# Absolute file URIs so the logged lineage doesn't depend on the notebook's CWD.
# MLflow registers two identical local sources and warns on every local path; harmless.
with warnings.catch_warnings():
    warnings.filterwarnings('ignore', message='The specified dataset source can be interpreted in multiple ways')
    train_ds = mlflow.data.from_pandas(df_train[raw_cols + [target]], source=Path(train_path).resolve().as_uri(), targets=target, name='green-2026-01')
    val_ds = mlflow.data.from_pandas(df_val[raw_cols + [target]], source=Path(val_path).resolve().as_uri(), targets=target, name='green-2026-02')

len(df_train), len(df_val)

(38088, 35319)

# Pipeline

In [4]:
def build_pipeline(regressor):
    preprocess = ColumnTransformer([
        ('num', StandardScaler(), ['trip_distance']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['PU_DO']),
    ])
    return make_pipeline(
        FunctionTransformer(add_features), 
        preprocess, 
        regressor
    )


def log_pred_plot(y_pred, y_true):
    fig, ax = plt.subplots()
    sns.histplot(y_pred, label='prediction', stat='density', ax=ax)
    sns.histplot(y_true, label='actual', stat='density', ax=ax)
    ax.legend()
    mlflow.log_figure(fig, 'pred_vs_actual.png')
    plt.close(fig)

# Linear Regression

In [5]:
with mlflow.start_run(run_name='linreg-baseline'):
    mlflow.set_tag('model_family', 'linear')
    mlflow.log_input(train_ds, context='training')
    mlflow.log_input(val_ds, context='validation')

    lr_model = build_pipeline(LinearRegression()).fit(X_train, y_train)
    y_pred = lr_model.predict(X_val)

    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric('val_rmse', rmse)
    log_pred_plot(y_pred, y_val)

    mlflow.sklearn.log_model(
        lr_model,
        name='linreg',
        signature=infer_signature(X_val, y_pred),
        input_example=X_val.head(5),
        **MODEL_LOG_KWARGS,
    )

rmse

7.097726072163113

# XGBoost

In [6]:
MAX_EVALS = 50
TOP_FRACTION = 0.1  # share of trials whose model gets logged

search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 30, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'n_estimators': scope.int(hp.quniform('n_estimators', 100, 1000, 1)),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -5, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:squarederror',
    'random_state': 42,
}


def objective(params):
    # Child run per trial: metrics only. Models are logged afterwards for the top trials,
    # since we only know which trials are "top" once the search is over
    with mlflow.start_run(nested=True) as run:
        mlflow.log_params(params)
        model = build_pipeline(xgb.XGBRegressor(**params)).fit(X_train, y_train)
        rmse = root_mean_squared_error(y_val, model.predict(X_val))
        mlflow.log_metric('val_rmse', rmse)
    return {'loss': rmse, 'status': STATUS_OK, 'run_id': run.info.run_id, 'params': params}


with mlflow.start_run(run_name='xgb-hyperopt'):
    mlflow.set_tag('model_family', 'xgboost')
    mlflow.log_input(train_ds, context='training')
    mlflow.log_input(val_ds, context='validation')

    trials = Trials()
    fmin(fn=objective,
         space=search_space,
         algo=tpe.suggest,
         max_evals=MAX_EVALS,
         trials=trials
    )

    n_top = max(1, math.ceil(len(trials.results) * TOP_FRACTION))
    top_results = sorted(trials.results, key=lambda r: r['loss'])[:n_top]

    # Refit each top trial (random_state is fixed, so it's the same model) and
    # attach it to that trial's own child run. Only the winner is registered.
    for rank, result in enumerate(top_results, start=1):
        with mlflow.start_run(run_id=result['run_id'], nested=True):
            model = build_pipeline(xgb.XGBRegressor(**result['params'])).fit(X_train, y_train)
            y_pred = model.predict(X_val)

            mlflow.set_tag('top_rank', rank)
            log_pred_plot(y_pred, y_val)

            info = mlflow.sklearn.log_model(
                model,
                name=f'xgb-top{rank}',
                params=result['params'],
                signature=infer_signature(X_val, y_pred),
                input_example=X_val.head(5),
                registered_model_name=REGISTERED_MODEL_NAME if rank == 1 else None,
                **MODEL_LOG_KWARGS,
            )
        if rank == 1:
            model_info = info

    best_params, rmse = top_results[0]['params'], top_results[0]['loss']
    mlflow.log_params(best_params)
    mlflow.log_metric('val_rmse', rmse)
    mlflow.set_tag('best_run_id', top_results[0]['run_id'])

best_params, rmse

100%|██████████| 50/50 [04:32<00:00,  5.46s/trial, best loss: 5.948702010883866]


Successfully registered model 'nyc-taxi-duration'.
Created version '1' of model 'nyc-taxi-duration'.


({'learning_rate': 0.123481687143061,
  'max_depth': 8,
  'min_child_weight': 2.023908103943982,
  'n_estimators': 998,
  'objective': 'reg:squarederror',
  'random_state': 42,
  'reg_alpha': 0.009389808722361843,
  'reg_lambda': 0.23847500361888035},
 5.948702010883866)

# Optuna search per model family

Each family gets a time-boxed Optuna study (one nested run per trial). The best trial is refit in the parent run, logged, registered under `REGISTERED_MODEL_NAME`, tagged with its family and aliased `best-<family>`, so it loads with `models:/nyc-taxi-duration@best-<family>`.

In [7]:
TIMEOUT = 120  # seconds of search per model family


def tune(family, make_regressor, timeout=TIMEOUT):
    def objective(trial):
        with mlflow.start_run(nested=True):
            model = build_pipeline(make_regressor(trial)).fit(X_train, y_train)
            rmse = root_mean_squared_error(y_val, model.predict(X_val))
            mlflow.log_metric('val_rmse', rmse)
        return rmse

    with mlflow.start_run(run_name=f'{family}-optuna'):
        mlflow.set_tag('model_family', family)
        mlflow.log_input(train_ds, context='training')
        mlflow.log_input(val_ds, context='validation')

        study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
        study.optimize(objective, timeout=timeout)

        # FixedTrial replays the best params through the same make_regressor; fixed seeds make it the same model
        model = build_pipeline(make_regressor(optuna.trial.FixedTrial(study.best_params))).fit(X_train, y_train)
        y_pred = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)

        mlflow.log_metric('val_rmse', rmse)
        mlflow.set_tag('n_trials', len(study.trials))
        log_pred_plot(y_pred, y_val)

        info = mlflow.sklearn.log_model(
            model,
            name=family,
            signature=infer_signature(X_val, y_pred),
            input_example=X_val.head(5),
            registered_model_name=REGISTERED_MODEL_NAME,
            **MODEL_LOG_KWARGS,
        )

    client = MlflowClient()
    version = info.registered_model_version
    client.set_model_version_tag(REGISTERED_MODEL_NAME, version, 'model_family', family)
    client.set_model_version_tag(REGISTERED_MODEL_NAME, version, 'val_rmse', f'{rmse:.4f}')
    client.set_registered_model_alias(REGISTERED_MODEL_NAME, f'best-{family}', version)
    return study.best_params, rmse


## Random Forest

In [8]:
tune('random_forest', lambda t: RandomForestRegressor(
    n_estimators=t.suggest_int('n_estimators', 50, 200),
    max_depth=t.suggest_int('max_depth', 5, 30),
    min_samples_leaf=t.suggest_int('min_samples_leaf', 1, 10),
    max_features=t.suggest_float('max_features', 0.1, 1.0),
    n_jobs=-1,
    random_state=42,
))

Registered model 'nyc-taxi-duration' already exists. Creating a new version of this model...
Created version '2' of model 'nyc-taxi-duration'.


({'n_estimators': 129,
  'max_depth': 29,
  'min_samples_leaf': 3,
  'max_features': 0.34209507623269525},
 6.069038971639749)

## Decision Tree

In [9]:
tune('decision_tree', lambda t: DecisionTreeRegressor(
    max_depth=t.suggest_int('max_depth', 3, 40),
    min_samples_split=t.suggest_int('min_samples_split', 2, 50),
    min_samples_leaf=t.suggest_int('min_samples_leaf', 1, 50),
    random_state=42,
))

Registered model 'nyc-taxi-duration' already exists. Creating a new version of this model...
Created version '3' of model 'nyc-taxi-duration'.


({'max_depth': 16, 'min_samples_split': 50, 'min_samples_leaf': 4},
 6.177879041989273)

## SVM

Kernel `SVR` is O(n²) in samples (~40k here), too slow for a minutes-long search, so this uses `LinearSVR`.

In [10]:
tune('svm', lambda t: LinearSVR(
    C=t.suggest_float('C', 1e-3, 10, log=True),
    epsilon=t.suggest_float('epsilon', 0.0, 2.0),
    loss=t.suggest_categorical('loss', ['epsilon_insensitive', 'squared_epsilon_insensitive']),
    max_iter=5000,
    random_state=42,
))

Registered model 'nyc-taxi-duration' already exists. Creating a new version of this model...
Created version '4' of model 'nyc-taxi-duration'.


({'C': 1.4382727819070547,
  'epsilon': 1.281680240953753,
  'loss': 'squared_epsilon_insensitive'},
 7.022684624950669)

## Neural Network (one hidden layer)

In [11]:
tune('mlp', lambda t: MLPRegressor(
    hidden_layer_sizes=(t.suggest_int('hidden_units', 16, 256, log=True),),
    alpha=t.suggest_float('alpha', 1e-5, 1e-1, log=True),
    learning_rate_init=t.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
    early_stopping=True,
    max_iter=200,
    random_state=42,
))

Registered model 'nyc-taxi-duration' already exists. Creating a new version of this model...
Created version '5' of model 'nyc-taxi-duration'.


({'hidden_units': 83,
  'alpha': 4.207988669606632e-05,
  'learning_rate_init': 0.00020511104188433984},
 7.079018810154734)

# Registry

## Import a Logistic Regression Model

## Import a XGboost Model